# Классификация эмоций по тексту: Злость, Нейтраль, Счастье
## Автономный Jupyter Notebook для решения задачи многоклассовой классификации

**Цель:** Разработать и обучить модель глубокого обучения для классификации текстов на 3 класса эмоций с использованием современных архитектур (трансформеры + механизм внимания + BiLSTM).

**Структура ноутбука:**
1. Сбор данных
2. Предобработка данных
3. Создание моделей
4. Оценка работоспособности
5. Интерпретация результатов

In [ ]:
# ============================================================================
# ЯЧЕЙКА 1: Автоматическая установка зависимостей
# ============================================================================
import sys
import subprocess
import importlib

REQUIRED_PACKAGES = [
    'torch', 'transformers', 'datasets', 'scikit-learn',
    'matplotlib', 'seaborn', 'numpy', 'pandas', 'tqdm'
]

def install_package(package_name):
    try:
        importlib.import_module(package_name)
        print(f"✓ {package_name} уже установлен")
        return True
    except ImportError:
        print(f"⚠ Установка {package_name}...")
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', package_name, '-q'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f"✓ {package_name} успешно установлен")
        return True

print("=" * 60)
print("ПРОВЕРКА И УСТАНОВКА ЗАВИСИМОСТЕЙ")
print("=" * 60)
for pkg in REQUIRED_PACKAGES:
    install_package(pkg)
print("\n✓ Все зависимости установлены/проверены")
print("=" * 60)

---
## ЭТАП 1: СБОР ДАННЫХ

### Описание этапа
Загружаем публичный датасет эмоций из HuggingFace и преобразуем в 3 класса:
- **anger** → "злость" (класс 0)
- **joy** → "счастье" (класс 1)
- **остальные** → "нейтраль" (класс 2)

При недоступности сети — fallback на синтетические данные.

In [ ]:
# ============================================================================
# ЯЧЕЙКА 2: Импорт библиотек и настройка окружения
# ============================================================================
import os, random, warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, BertTokenizer, BertModel
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Фиксация random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Устройство: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ Работа на CPU (рекомендуется для быстрого демо <5 мин)")